#Install Dependencies

In [1]:
%%capture
!pip install unsloth
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes datasets

#Mount the Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

print("Google Drive mounted. You can now access your files via '/content/drive/MyDrive/'")

Mounted at /content/drive
Google Drive mounted. You can now access your files via '/content/drive/MyDrive/'


#Load the Model from the Backup

In [3]:
from unsloth import FastLanguageModel
import torch

print("Loading Base Model + LoRA Adapters...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "/content/drive/MyDrive/lora_backup", # Points to your uploaded folder
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)

# Switch to 2x faster inference mode
FastLanguageModel.for_inference(model)
print("Model Ready for Inference!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Loading Base Model + LoRA Adapters...
==((====))==  Unsloth 2026.8.10: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load /content/drive/MyDrive/lora_backup as a legacy tokenizer.
Unsloth 2026.8.10 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


Model Ready for Inference!


#The Inference Generator

In [7]:
def clean_sinhala_text(noisy_text):
    # The exact Alpaca prompt used during training
    alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
You are a Sinhala ASR correction system. Fix the spelling and grammar of the noisy input text while preserving all numbers and context.

### Input:
{input}

### Response:
"""

    # Format the prompt with the noisy text
    formatted_prompt = alpaca_prompt.format(input=noisy_text)
    inputs = tokenizer([formatted_prompt], return_tensors="pt").to("cuda")

    # Generate the output
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        use_cache=True,
        temperature=0.1, # Low temperature for accurate correction
        pad_token_id=tokenizer.eos_token_id
    )

    # Decode and extract the response
    full_output = tokenizer.batch_decode(outputs, skip_special_tokens=False)[0]

    try:
        # Split at the response trigger
        generated_text = full_output.split("### Response:\n")[-1]
        # Clean up hidden tokens
        generated_text = generated_text.replace(tokenizer.eos_token, "").strip()
        generated_text = generated_text.replace("<|eot_id|>", "").strip()
    except Exception:
        generated_text = "Error: Could not parse output."

    return generated_text

In [5]:
# ==========================================
# TEST IT HERE
# ==========================================
test_text = "ආයුබෝවන් ම තුෂාරි මට පුළුවනි ඔබට සහාය වන්න ආයුබෝවන් මිස් මගේ මේ නම්බර් එක එක කනෙක්ෂන් එක බව වෙලා තියෙනවා හ්ම් ඒක පොඩ්නල් රීකනෙක්ට් කරගන්න ඕනේ ම අද වසට පේමන්ට් එක කරන්න පුළුවන් නම තියෙන්නේ කොහොමද එම්අාර් රැඳී ක් චෙක් කරලා බලන්නම් ඇමතුමේ රැඳී ිටින්න හරි ඇමතුමේ රැඳී සිටින්න සර් ඕකේ බිල් එක තියෙනවා යි සත් ඇය ඇතුළ පේමන්ට් එක කරන්න ඕනේ සර් ඇමතුමේ රැ සිටින්න හරි හරි කනෙක්ට් වෙන්නේ නෑ සර් පේමන්ට් එක කරන්න ඕනේ ආ සාමාන්යෙන් කීයක් වගේද පේමන්ට් එක දාන්න තියෙන්නේ අ ඉන්ටරනෙට් නම් කනෙක්ට් වුනාවගේ ක් ගෙ වෙන්න ඕනේ සර් ා ඉන්ට්නෙට් කනෙක්ට් වෙනවා මොකද ස්ලෝ කරලා තියෙන්නේ රයිට් ඉන්ටර්නෙට්ට් යන්න බැරි කමක් නැ සර්ට ඉන්ටරනෙට් යන්න පුළුවන් ස්ලෝ කරලා තියෙන්නේ ස්ලෝ කරලා තියෙන්නේ ආ ඕකේඅම පේමන්ට් එක කරන්නම් ක් වගේ නේ ඔව් ආ ඕකේ ම දැන් දාන්නම් හරි හරි තැන්ක් යු වෙනත් යමක් දැනගන්න අවශ්යිද නැහැ මිස් මා ලබා දුන් සේවයා ඇගයීම සඳහා රැඳී සිටින්න එස්එල්ටී මොබිටෙල් ඇමතුවට ස්තූතියි සුභ දවසක් සුභ දවස්යක්"

print("Original (Noisy) :", test_text)
print("--------------------------------------------------")
print("Cleaned Output   :", clean_sinhala_text(test_text))

Both `max_new_tokens` (=256) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Original (Noisy) : ආයුබෝවන් ම තුෂාරි මට පුළුවනි ඔබට සහාය වන්න ආයුබෝවන් මිස් මගේ මේ නම්බර් එක එක කනෙක්ෂන් එක බව වෙලා තියෙනවා හ්ම් ඒක පොඩ්නල් රීකනෙක්ට් කරගන්න ඕනේ ම අද වසට පේමන්ට් එක කරන්න පුළුවන් නම තියෙන්නේ කොහොමද එම්අාර් රැඳී ක් චෙක් කරලා බලන්නම් ඇමතුමේ රැඳී ිටින්න හරි ඇමතුමේ රැඳී සිටින්න සර් ඕකේ බිල් එක තියෙනවා යි සත් ඇය ඇතුළ පේමන්ට් එක කරන්න ඕනේ සර් ඇමතුමේ රැ සිටින්න හරි හරි කනෙක්ට් වෙන්නේ නෑ සර් පේමන්ට් එක කරන්න ඕනේ ආ සාමාන්යෙන් කීයක් වගේද පේමන්ට් එක දාන්න තියෙන්නේ අ ඉන්ටරනෙට් නම් කනෙක්ට් වුනාවගේ ක් ගෙ වෙන්න ඕනේ සර් ා ඉන්ට්නෙට් කනෙක්ට් වෙනවා මොකද ස්ලෝ කරලා තියෙන්නේ රයිට් ඉන්ටර්නෙට්ට් යන්න බැරි කමක් නැ සර්ට ඉන්ටරනෙට් යන්න පුළුවන් ස්ලෝ කරලා තියෙන්නේ ස්ලෝ කරලා තියෙන්නේ ආ ඕකේඅම පේමන්ට් එක කරන්නම් ක් වගේ නේ ඔව් ආ ඕකේ ම දැන් දාන්නම් හරි හරි තැන්ක් යු වෙනත් යමක් දැනගන්න අවශ්යිද නැහැ මිස් මා ලබා දුන් සේවයා ඇගයීම සඳහා රැඳී සිටින්න එස්එල්ටී මොබිටෙල් ඇමතුවට ස්තූතියි සුභ දවසක් සුභ දවස්යක්
--------------------------------------------------
Cleaned Output   : ආයුබෝවන්, මම තුෂාරි. මට පුළුවනි ඔබට 

In [8]:
# ==========================================
# TEST IT HERE
# ==========================================
test_text = "ඕය ම පිනකයි මට පුවන් සයවන්නහෙලෝ න්ටඉන්න හෙලෝ පේඔව් කියන්න සර් හෙලෝ ඔව් මගේ රවුටර බිල් එක මට දැන් මාස දෙක් විතර ගෙවාගන්න බැරි වුනා මේ ම වෙනදට බිල් එක පේ කරන්නේ විදිහට ඒ කරගන්න විදිහන්නේනෑහැ ඔන්ලයින් පමේ පේමන්ට් එකට බිල් වීව් එක කියරලා තමයි යන්නේ ම මට දැන් ඒකේ රවුටර් බිල් එකේ මවුන්ට් එක හරියට දැනගන්නයි කොහොමද මේ ඔන්ලයින් පේමන්ට් එක කරගන්න පුළුවන් විදිහක් පලිය කරගන්න තමයි ගත්තේ හරි ඔය සර්ගේ අදාළ එස් එල්ටී කනෙක්ෂන් එකේ නම්බර් එක කියන්න බංදු දයි හරි යි දෙකයි හරි අයි ර කරුණාකර ඇතුමේ රැඳී ඉන්නඕකරැඳිටාට ස්තූතියි සර් මෙතන කනෙක්ෂන් හිමිකරුගේ නම කියන්න මිස්ටර් විතර් සිංහදඔව් එතකොට සර් කොව වෙනදා කොහොමද පේමන්ට් එක කරන්නේ සර් ලයින් එකට දැන ටරනම් ලයින් එක ස්පේන්ඩ් වෙලා තියෙන්නේ ම සාමාන්යෙන් කරගෙන ආවේ මගේ වීව් කරනවා ඔන්ලයින් බිල් වීව් එකෙන් බලා ඒක එවලේ ඒ කියන්නේ ඔන්ලයින් විදිහට ප්රොසීඩ් කරන එක තමයි පේමන්ට් එක කරේ ම එහෙම තමයි සෑහෙන කාලයක් රගෙන මේ මාස දෙක් තිස්සේ මට ඒක මේ වීව් වෙන්නේ නෑ කොච්චර කරත් බිල් එක වීව් වෙන්නේ නැහැ මේ ඒ ප්රශ්නෙ නිසා තමයි ඇත්තකට ම මේ ගෙවුනේ නැත්තේ හරි ම සර්ට ලින්ක් දෙක් එවන්න නම් බිල් වීව් ලින්ක් එකයි ඒ වගේම පේ ඔන්ලයින් ලින්ක් එකුයි සර්ගේ මොබයල් කන්ටැක්ට් නම්බර් එක කියන්න මේ කතා කරන අංකයට ගන්න පුළුවන්ද ඔහ් මේ නම්බර් එකට පුළුවන්රන ම්බ්යට හරි දැන් ඔන්වන් නම්බර් වීප කවුන්ට් නම්බර් එක යටතේ සර්ගේ තියෙන්නේ කනෙක්ෂන් එක ෆූජී කනෙක්ෂන් එක් නේද ඔව් ෂ ඉන්ටර්නෙට් එකට විතරක් භාවිතා කරන එක සර්විස් එක් හරි ඇමතුවට රැඳී න්නේ සර් ඔවමතුම රැඳී න්නේ සර් ලයින් එක ඉන්නකේ හරි සර් රැනටිසටාට ස්තියි ම දැන් ලින්ක් දෙක් සර්ට එවලා තියෙනවා එක් පේ ඉන්ස්ටන්ලි කියන ලින්ක් එක අනිත් එක ේඔමේවිව් කරගන්න ලින්ක් එක සර් මේ දෙකම භාිවිනවන් එකේ කරන්න පුළුවන්ද කියලා හරි සර් වෙනත් යමක් දැනගන්න අවශ්යද දැනට ම මේකේ ප්රොසීඩ් කරලා බලන්නම් කොහොමද බිල් එක පෙන්න්නේ මවුන්ට් එක මොක්ද සර් බිල්මවුන්ට් එක මේ වෙනකොට කීයද පෙන්න් බිල් මවුන්ට් එක දැනට් පෙන්නුම් කරනවා මේක අප්ඩේටඩ් වෙලා තියෙන්නේ නි මාසේ දක්වා් දැන් ජනවාරි මාසේ පළවෙනිදත් බිල් එක ඉෂූ වෙලා තියෙනවා ක් සවි අටක් කේ හරි සර් තැන්ක් යූ එස් එල්ටී මොබිටෙල් ඇමතුවාට ස්තූතියි සුභ දවක් ඇමතුමඇගයසකන්"

print("Original (Noisy) :", test_text)
print("--------------------------------------------------")
print("Cleaned Output   :", clean_sinhala_text(test_text))

Both `max_new_tokens` (=512) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Original (Noisy) : ඕය ම පිනකයි මට පුවන් සයවන්නහෙලෝ න්ටඉන්න හෙලෝ පේඔව් කියන්න සර් හෙලෝ ඔව් මගේ රවුටර බිල් එක මට දැන් මාස දෙක් විතර ගෙවාගන්න බැරි වුනා මේ ම වෙනදට බිල් එක පේ කරන්නේ විදිහට ඒ කරගන්න විදිහන්නේනෑහැ ඔන්ලයින් පමේ පේමන්ට් එකට බිල් වීව් එක කියරලා තමයි යන්නේ ම මට දැන් ඒකේ රවුටර් බිල් එකේ මවුන්ට් එක හරියට දැනගන්නයි කොහොමද මේ ඔන්ලයින් පේමන්ට් එක කරගන්න පුළුවන් විදිහක් පලිය කරගන්න තමයි ගත්තේ හරි ඔය සර්ගේ අදාළ එස් එල්ටී කනෙක්ෂන් එකේ නම්බර් එක කියන්න බංදු දයි හරි යි දෙකයි හරි අයි ර කරුණාකර ඇතුමේ රැඳී ඉන්නඕකරැඳිටාට ස්තූතියි සර් මෙතන කනෙක්ෂන් හිමිකරුගේ නම කියන්න මිස්ටර් විතර් සිංහදඔව් එතකොට සර් කොව වෙනදා කොහොමද පේමන්ට් එක කරන්නේ සර් ලයින් එකට දැන ටරනම් ලයින් එක ස්පේන්ඩ් වෙලා තියෙන්නේ ම සාමාන්යෙන් කරගෙන ආවේ මගේ වීව් කරනවා ඔන්ලයින් බිල් වීව් එකෙන් බලා ඒක එවලේ ඒ කියන්නේ ඔන්ලයින් විදිහට ප්රොසීඩ් කරන එක තමයි පේමන්ට් එක කරේ ම එහෙම තමයි සෑහෙන කාලයක් රගෙන මේ මාස දෙක් තිස්සේ මට ඒක මේ වීව් වෙන්නේ නෑ කොච්චර කරත් බිල් එක වීව් වෙන්නේ නැහැ මේ ඒ ප්රශ්නෙ නිසා තමයි ඇත්තකට ම මේ ගෙවුනේ නැත්තේ හරි ම සර්ට ලි